In [ ]:
%matplotlib qt
import time
import csv
from collections import deque
import serial
import serial.tools.list_ports
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# ---------------- Configuración ----------------
PUERTO = "COM5"
BAUDIOS = 115200
VENTANA_S = 10
ARCHIVO_CSV = f"angulos_{time.strftime('%Y%m%d_%H%M%S')}.csv"
ARTICULACIONES = ["hombro", "codo", "carpo"]

def parsear(linea):
    datos = {}
    for parte in linea.split(","):
        if ":" not in parte:
            return None
        nombre, valor = parte.split(":", 1)
        try:
            datos[nombre.strip()] = float(valor)
        except ValueError:
            return None
    return datos if all(a in datos for a in ARTICULACIONES) else None

# ---------------- Conexión ----------------
try:
    ser.close()          # si re-ejecutas la celda, cierra la conexión anterior
except NameError:
    pass

ser = serial.Serial(PUERTO, BAUDIOS, timeout=0.05)
time.sleep(2)            # la ESP32 se reinicia al abrir el puerto
print(f"Conectado a {PUERTO}")
print(ser.read(ser.in_waiting).decode(errors="ignore"))   # muestra mensajes de calibración


In [ ]:
# ---------------- Gráfica ----------------
ser.reset_input_buffer()          # descarta datos viejos acumulados

archivo = open(ARCHIVO_CSV, "w", newline="")
escritor = csv.writer(archivo)
escritor.writerow(["t_s"] + ARTICULACIONES)

t0 = time.time()
tiempos = deque()
series = {a: deque() for a in ARTICULACIONES}

fig, ax = plt.subplots(figsize=(10, 5))
lineas = {a: ax.plot([], [], label=a)[0] for a in ARTICULACIONES}
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Ángulo (°)")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")

def actualizar(_):                # <- ESTO es lo que actualiza la señal
    while ser.in_waiting:
        linea = ser.readline().decode(errors="ignore").strip()
        datos = parsear(linea)
        if datos is None:
            continue
        t = time.time() - t0
        tiempos.append(t)
        for a in ARTICULACIONES:
            series[a].append(datos[a])
        escritor.writerow([f"{t:.3f}"] + [f"{datos[a]:.2f}" for a in ARTICULACIONES])

    while tiempos and tiempos[-1] - tiempos[0] > VENTANA_S:
        tiempos.popleft()
        for a in ARTICULACIONES:
            series[a].popleft()

    if tiempos:
        for a in ARTICULACIONES:
            lineas[a].set_data(tiempos, series[a])
        ax.set_xlim(max(0, tiempos[-1] - VENTANA_S), max(VENTANA_S, tiempos[-1]))
        todos = [v for a in ARTICULACIONES for v in series[a]]
        ax.set_ylim(min(todos) - 5, max(todos) + 5)
    return list(lineas.values())

def al_teclear(evento):
    if evento.key == "c":
        ser.write(b"c")

fig.canvas.mpl_connect("close_event", lambda _: archivo.close())
fig.canvas.mpl_connect("key_press_event", al_teclear)

anim = FuncAnimation(fig, actualizar, interval=30, cache_frame_data=False)   # <- y ESTO la llama cada 30 ms
plt.show()

In [ ]:
ser.close()
print("Puerto cerrado")